# Day 1 수강생 실습용 노트북

이 노트북은 강사용 노트북에서 **핵심 Python 코드 일부를 비워 둔 실습용 자료**입니다.

프롬프트, 알람 데이터, 매뉴얼 데이터, 조회 결과 예시, 레포트 예시 데이터는 그대로 제공됩니다.

수강생은 `TODO`로 표시된 부분의 Python 코드를 직접 작성하면서 제조 AI Agent의 기본 흐름을 연습합니다.

> 셀은 위에서 아래로 순서대로 실행하세요. 일부 칸이 비어 있어 결과가 완전하지 않을 수 있으며, TODO를 채우면 완성됩니다.

# Day1. 제조 AI Agent 기초 실습 노트북

이 노트북은 제조 현장의 알람 상황을 다루는 **가장 단순한 AI Agent**의 흐름을 배우기 위한 강의용 노트북입니다.

## 이 노트북의 목적
- Python 코드가 위에서 아래로 실행되는 흐름을 이해합니다.
- 제조 알람 데이터를 함수로 처리하는 과정을 따라가 봅니다.
- "입력을 보고 다음 행동을 정하는" Agent의 기본 구조를 이해합니다.

## 읽기 전에 꼭 알아둘 점
1. 이 노트북은 `src/day1` 원본 Python 파일들을 **바탕으로** 만들었습니다.
2. 원본 파일 **전체를 복사한 것이 아니라**, 학습에 꼭 필요한 핵심 코드만 추출했습니다.
3. 초보자가 이해하기 쉽도록 **일부 코드는 단순화**했습니다. (예: pandas → 순수 Python, 파일 입출력 제거)
4. 각 단원 시작에는 **어떤 원본 파일에서 가져왔는지**를 표시합니다.
5. 이 노트북은 `src/day1` 원본 파일을 **수정하지 않습니다.**
6. 복잡한 흐름(LangGraph, Mustache 템플릿, .env 설정 등)은 강의 이해를 위해 **단순화**했습니다.
7. **외부 API Key 없이** 실행됩니다. (실제 LLM 대신 교육용 Mock 함수를 사용)
8. Code 셀은 **위에서 아래로 순서대로** 실행하도록 설계했습니다. 중간 셀만 실행하면 오류가 날 수 있습니다.

> 사용하는 데이터는 모두 교육용 가상 시나리오(**DisplayEdu Fab**)이며, 실제 사내 데이터·시스템과 무관합니다.

## LLM 호출 실습 안내

이 노트북에는 주요 실습 블록 뒤에 **LLM 해설 셀**이 들어 있습니다.

각 LLM 해설 셀은 지금까지 실행한 코드와 결과를 바탕으로
"무슨 일이 일어났는지", "초보자가 이해해야 할 핵심이 무엇인지"를 LLM에게 설명하게 합니다.

### 이 노트북이 LLM과 통신하는 방법
- 이 노트북은 `from openai import OpenAI`를 사용해 LLM과 통신합니다.
- 여기서 **OpenAI**는 꼭 OpenAI 회사의 모델만 의미하는 것이 아닙니다.
  **NVIDIA API**처럼 OpenAI와 **같은 요청 형식(OpenAI-compatible)** 을 지원하는 LLM 서버도 같은 방식으로 호출할 수 있습니다.
- 코드에서는 `OpenAI` 클래스를 사용하지만, `base_url`을 NVIDIA 주소로 두면 실제로는 NVIDIA가 제공하는 LLM API로 요청을 보냅니다.

### 미리 준비할 것
- **openai 패키지**: 설치되어 있지 않다면 터미널에서 `pip install openai` 를 실행하세요.
  (이 노트북은 코드 셀에서 자동으로 설치하지 않습니다. 설치는 직접 터미널에서 합니다.)
- **API Key (환경변수)**: API Key는 LLM 서버에 접속하기 위한 **비밀번호와 비슷**합니다.
  그래서 코드에 직접 적지 않고 **환경변수**에 저장합니다. 이 노트북은 아래 환경변수를 사용합니다.
  - `NVIDIA_API_KEY` — LLM 접속용 키 (필수, 없으면 교육용 안내 응답이 대신 표시됩니다)
  - `NVIDIA_BASE_URL` — LLM 서버 주소 (기본값 `https://integrate.api.nvidia.com/v1`)
  - `NVIDIA_MODEL` — 사용할 모델명 (기본값 `nvidia/llama-3.1-nemotron-70b-instruct`)

### 공통 함수 `ask_llm(prompt, title=...)`
- 수강생이 매번 복잡한 LLM 통신 코드를 작성하지 않도록, 노트북 안에 공통 함수 `ask_llm`을 직접 선언했습니다.
- 수강생은 **prompt만 만들고 `ask_llm(prompt)` 를 호출**하면 됩니다. 응답은 문자열로 돌아오며, `display(Markdown(...))` 으로 화면에 보여 줍니다.
- `title`은 어떤 설명을 요청하는지 **구분하기 위한 이름**이며, 이번 노트북에서는 `[LLM 요청] 제목` 형태로 간단히 출력됩니다.
- **`NVIDIA_API_KEY` 가 없어도** `ask_llm` 이 교육용 안내 문구를 돌려주므로 노트북 흐름이 끊기지 않습니다.

> 아래 **두 개의 공통 유틸리티 셀을 먼저 실행**해야 LLM 해설 셀들이 동작합니다.

이번 노트북의 LLM 해설 prompt들은 `templates/day1`의 `*prompt.mustache` 구조를 참고해 작성했고,
모두 가상의 디스플레이 박막 제조사 **DisplayEdu ThinFilm Co.** 상황을 기준으로 합니다.

In [ ]:
# === [공통 유틸리티 1/2] LLM 호출 준비 (LLM 해설 셀보다 먼저 한 번 실행) ===
# import: 외부에 이미 만들어져 있는 기능을 코드로 가져오는 문장입니다.
import os                                       # os: 환경변수 등 운영체제 기능을 다루는 기본 모듈
import json                                     # json: 데이터를 사람이 읽기 쉬운 문자열로 바꾸는 모듈
from openai import OpenAI                        # OpenAI: LLM 서버에 요청을 보내는 클라이언트(통신 도구)
from IPython.display import display, Markdown    # display/Markdown: 결과를 노트북에 보기 좋게 출력하는 도구

# 이 노트북은 OpenAI 클래스로 LLM과 통신합니다. OpenAI-compatible(= OpenAI와 같은 요청 형식)이라,
# base_url만 NVIDIA 주소로 바꾸면 NVIDIA API 같은 다른 LLM 서버도 똑같은 코드로 호출할 수 있습니다.


def ask_llm(prompt, title="LLM 응답"):
    """prompt를 LLM에 보내고, 응답 텍스트(문자열)를 돌려주는 공통 함수입니다.
    각 실습 셀에서는 prompt만 만들고 이 함수를 호출하면 됩니다.
    prompt = LLM에게 전달하는 질문/지시문(업무 지시서), title = 어떤 설명을 요청하는지 구분하는 이름입니다."""
    print(f"[LLM 요청] {title}")                 # f-string: 문장 안 {title}에 변수 값을 끼워 넣어 화면에 표시

    # os.getenv: 코드에 직접 쓰지 않은 "환경변수" 값을 읽어 오는 함수입니다.
    # API Key는 LLM 서버에 접속하기 위한 비밀번호와 비슷해, 코드에 적지 않고 환경변수에 보관합니다.
    api_key = os.getenv("NVIDIA_API_KEY")
    base_url = os.getenv("NVIDIA_BASE_URL", "https://integrate.api.nvidia.com/v1")  # base_url = LLM 서버 주소
    model = os.getenv("NVIDIA_MODEL", "nvidia/llama-3.1-nemotron-70b-instruct")     # model = 사용할 LLM 모델 이름

    # if 조건문: API Key가 없으면 실제 호출을 생략하고 교육용 안내 문구를 돌려줍니다. (노트북 흐름이 끊기지 않습니다.)
    if not api_key:
        return (
            "## (교육용 안내 응답)\n"
            "- 현재 `NVIDIA_API_KEY` 환경변수가 설정되어 있지 않아, 실제 LLM 호출은 생략되었습니다.\n"
            "- 수업에서는 이 부분에 LLM이 생성한 쉬운 설명이 표시된다고 이해하면 됩니다.\n"
            "- API Key는 LLM 서버에 접속하기 위한 비밀번호와 비슷해서, 코드에 직접 적지 않고 환경변수에 저장합니다."
        )

    # ----- 여기부터 수강생이 작성합니다 (API Key가 있을 때 실제 LLM을 호출하는 부분) -----
    # TODO 1: OpenAI(api_key=api_key, base_url=base_url) 로 client(통신 도구)를 만드세요.
    # TODO 2: client.chat.completions.create(...) 로 LLM을 호출하세요.
    #         - model=model 을 넣고
    #         - messages 에는 system 메시지(LLM의 역할 정의)와 user 메시지(content=prompt)를 넣으세요.
    #         - temperature=0.2 정도로 두면 일관된 답이 나옵니다.
    # TODO 3: 응답에서 답변 본문 텍스트를 꺼내 반환하세요. (힌트: response.choices[0].message.content)
    # 힌트: messages=[{"role": "system", "content": "...역할 설명..."}, {"role": "user", "content": prompt}]
    # 여기에 LLM 호출 코드를 작성하세요.
    llm_response = ""          # 기본값: 아직 작성 전이면 빈 문자열을 돌려줍니다.
    return llm_response


In [ ]:
# === [공통 유틸리티 2/2] 긴 데이터 줄이기 도우미 ===
# preview_text: 긴 텍스트를 LLM에 보낼 때 prompt가 너무 길어지지 않도록 앞부분만 잘라 주는 함수입니다.
def preview_text(value, max_chars=1500):
    # 문자열이 아니면(dict/list 등) 보기 좋은 JSON 문자열로 바꿉니다.
    text = value if isinstance(value, str) else json.dumps(value, ensure_ascii=False, indent=2, default=str)
    # 너무 길면 앞부분만 남기고 나머지는 생략 표시를 붙입니다.
    return text if len(text) <= max_chars else text[:max_chars] + "\n...(이하 생략)"

## src/day1 원본 파일별 목적 요약표

아래는 이 노트북이 참조한 `src/day1`(및 LLM 호출에 쓰이는 `src/llm`) 파일들입니다.

| 원본 파일 | 원본 파일의 목적 | 노트북 반영 방식 | 단순화 여부 |
|---|---|---|---|
| `first_sample_run.py` | 알람 로그·매뉴얼을 읽어 LLM으로 정리하는 첫 실행 예제 | 데이터 준비·로그 조회/요약 단원의 핵심으로 반영 | 단순화 (pandas→순수 Python, 파일 입출력 제거) |
| `simple_chain_starter.py` | 여러 단계를 순서대로 잇는 Chain 흐름 예제 | Chain(매뉴얼 검색→Prompt→LLM) 단원으로 반영 | 단순화 (Mustache 템플릿→f-string) |
| `mini_graph_runner.py` | State/Node/Edge/조건부 분기(LangGraph) 예제 | State/Node/Graph 기초 단원으로 반영 | 단순화 (LangGraph→순수 Python 함수) |
| `day1_agent_v0_template.py` | 위 개념을 합친 가장 단순한 Agent v0 | Agent v0 판단 함수 단원으로 반영 | 단순화 (LangGraph 그래프→함수 하나) |
| `src/llm_client.py`, `src/llm/mock_llm.py` | mock/local/cloud LLM 호출의 공통 창구와 교육용 Mock 응답 | Mock LLM 단원에서 개념만 반영 | 단순화 (provider 분기·API 호출 제거, 작은 Mock 함수) |
| `first_sample_run_streamlit_app.py` | 첫 실행을 웹 화면으로 보여주는 버전 | 참고 파일 (핵심 실습 제외) | — |
| `simple_chain_streamlit_app.py` | Chain을 웹 화면으로 보여주는 버전 | 참고 파일 (핵심 실습 제외) | — |
| `mini_graph_streamlit_app.py` | Mini Graph를 웹 화면으로 보여주는 버전 | 참고 파일 (핵심 실습 제외) | — |
| `day1_agent_v0_streamlit_app.py` | Agent v0를 웹 화면으로 보여주는 버전 | 참고 파일 (핵심 실습 제외) | — |
| `__init__.py` | 폴더를 패키지로 인식시키는 표시용 파일 | 제외 (실행 코드 없음) | — |

> Streamlit 화면 버전(`*_streamlit_app.py`) 4개는 CLI 버전과 **로직이 같고 화면 표시만 다르기** 때문에, 개념 학습에는 CLI 흐름이 더 적합하여 참고 파일로만 소개합니다.

## 1단계. Python 기본 실행 흐름 이해

본격적인 Agent 실습 전에, Python 코드가 **위에서 아래로 한 줄씩** 실행된다는 점을 먼저 확인합니다.

- 앞 셀에서 만든 변수와 함수는 뒤 셀에서 그대로 사용할 수 있습니다.
- 그래서 이 노트북은 **반드시 위에서부터 순서대로** 실행해야 합니다.

아래 셀을 실행해 보세요.

In [ ]:
# 변수(= 값을 담아 두는 이름표)에 문자열 값을 저장합니다.
# 한 번 담아 두면 다음 줄/다음 셀에서도 그 이름으로 값을 다시 꺼내 쓸 수 있습니다.
company = "DisplayEdu ThinFilm Co."   # 교육용 가상 디스플레이 박막 제조사명
line_id = "EDU-TF-LINE-07"            # 교육용 가상 박막 제조 라인명

# print()는 변수에 담긴 값을 화면에 찍어 중간 결과를 확인하는 출력 함수입니다.
print("회사:", company)
print("라인:", line_id)

## LLM 해설: Python 기본 실행 흐름

> 이 셀은 선택 실행 셀입니다. LLM API 설정이 되어 있지 않거나 강의 시간이 부족하면 건너뛰어도 됩니다.

이번 LLM 해설에서는 다음 관점으로 설명을 받습니다.
- 변수에 값을 저장한다는 의미
- print()로 결과를 확인하는 이유
- 앞에서 만든 값을 뒤 셀에서 다시 쓸 수 있다는 점
- 박막 제조 AI Agent 실습에서 실행 순서가 중요한 이유

이 prompt는 `templates/day1`의 `*prompt.mustache` 구조(역할·가상 데이터 안내·입력 정보·작성 요청·제약 조건)를 참고해 만들었으며, 노트북에서는 mustache 원문을 그대로 보여주지 않고 가상의 디스플레이 박막 제조 상황으로 단순화했습니다.

In [ ]:
# 앞에서 정의한 변수(company, line_id)를 LLM 해설용 입력으로 정리합니다.
python_flow_context = preview_text(f"company={company}, line_id={line_id}")

prompt = f"""# 역할
당신은 가상의 디스플레이 박막 제조사 'DisplayEdu ThinFilm Co.'의 AI Agent 교육 강사입니다.

# 교육용 가상 데이터 안내
이 실습은 실제 회사 데이터가 아니라 교육용 가상 박막 제조 시나리오입니다.
실제 사내 데이터·설비명·라인명·알람 코드·공정 조건과 무관합니다.

# 실습 블록
Python 코드가 위에서 아래로 실행되며, 박막 제조 라인 정보를 변수에 담는 기본 흐름입니다.

# 입력 정보
{python_flow_context}

# 작성 요청
초보자도 이해하도록 5~8줄로 짧게 설명해 주세요.
1. company가 가상의 박막 제조 회사명을 담는 변수라는 점
2. line_id가 교육용 박막 제조 라인을 나타낸다는 점
3. 앞에서 만든 변수를 뒤 셀에서 다시 쓸 수 있다는 점
4. 이런 기본 실행 흐름이 이후 박막 제조 AI Agent 실습의 출발점이 된다는 점

# 제약 조건
- 실제 현장 조치 지시처럼 단정하지 마세요.
- 입력에 없는 내용을 추측하지 마세요.
- "확인할 수 있습니다", "추가 확인이 필요합니다" 같은 교육용 표현을 사용하세요.
- 실제 회사명·설비 제조사명·공정 레시피·품질 기준은 넣지 마세요.
- 긴 보고서가 아니라 짧은 강의 해설로 작성하세요."""

# TODO 1: ask_llm 함수를 호출해 위에서 만든 prompt에 대한 LLM 응답을 받으세요.
# TODO 2: title 인자에 이 설명이 무엇인지 알 수 있는 제목을 넣으세요.
# TODO 3: display(Markdown(...)) 으로 응답을 화면에 보기 좋게 출력하세요.
# 힌트: llm_explanation_python_flow = ask_llm(prompt, title="...")
#       display(Markdown(llm_explanation_python_flow))
# 여기에 LLM 호출 코드를 작성하세요.
llm_explanation_python_flow = None


## 2단계. 제조 상황 예제 데이터 준비

**원본 파일**
- `src/day1/first_sample_run.py`

**원본 파일의 목적**
- 사용자 요청(JSON), 알람 로그(CSV), 매뉴얼(Markdown)을 읽어 LLM에게 정리를 맡기는 "첫 실행" 예제입니다.

**이 단원에서 학습할 핵심**
- Agent가 다루는 입력 데이터가 어떤 모양인지 이해합니다. (요청 / 로그 / 매뉴얼)

**노트북에서 사용한 핵심 코드**
- 원본의 입력 데이터를 노트북 변수(dict, list, 문자열)로 직접 만들어 사용합니다.

**노트북에서 단순화한 부분**
- 원본은 `data/sample_query.json`, `data/sample_alarm_logs.csv`, `docs/alarm_manual.md` 파일을 읽습니다.
- 노트북에서는 파일 읽기를 제외하고, 같은 구조의 데이터를 셀에 직접 작성했습니다.

**단순화한 이유**
- 실행 위치나 파일 경로에 상관없이 누구나 바로 실행할 수 있게 하기 위해서입니다.

**원본 파일을 나중에 볼 때 추가로 확인할 부분**
- `json.loads(...)`, `pandas.read_csv(...)`, `Path(...).read_text(encoding="utf-8-sig")`로 실제 파일을 읽는 부분을 확인하세요.

In [ ]:
# 딕셔너리(dict): "이름표(key)"와 "값(value)"을 짝으로 저장하는 구조입니다. (예: "equipment_id" → "TF-CVD-01")
# 아래는 사용자가 Agent에게 보낸 요청을 dict로 표현한 것이며, 모두 제조 설비에서 발생했다고 가정한 교육용 가상 데이터입니다.
# 원본 first_sample_run.py에서는 이 값을 sample_query.json 파일(읽기 쉬운 key-value 형식인 JSON)에서 읽습니다.
sample_query = {
    "user_query": "TF-CVD-01에서 ALM-TEMP-402가 반복 발생했습니다. 박막 증착 공정 관점에서 확인할 항목을 정리해 주세요.",
    "line_id": "EDU-TF-LINE-07",
    "process_name": "박막 증착 공정",
    "equipment_id": "TF-CVD-01",     # 가상 박막 증착(CVD) 설비 ID
    "alarm_code": "ALM-TEMP-402",    # 가상 챔버 온도 편차 알람
}
# dict에서 값을 꺼낼 때는 sample_query["키이름"] 처럼 대괄호 안에 이름표를 적습니다.
print(sample_query["equipment_id"], "/", sample_query["alarm_code"])

In [ ]:
# 리스트(list): 여러 값을 순서대로 담는 상자입니다. 대괄호 [ ] 안에 항목을 쉼표로 나열합니다.
# 아래 alarm_logs는 알람 한 건(dict)을 여러 개 담은 리스트이며, 제조 설비에서 발생했다고 가정한 교육용 가상 데이터입니다.
# 원본에서는 CSV 파일(표)로 읽지만, 여기서는 dict 목록으로 직접 표현했습니다. 각 줄(dict)이 알람 한 건입니다.
alarm_logs = [
    {"timestamp": "2026-05-20 09:10", "equipment_id": "TF-CVD-01", "alarm_code": "ALM-TEMP-402", "severity": "WARNING", "repeat_count": 2},
    {"timestamp": "2026-05-20 09:40", "equipment_id": "TF-CVD-01", "alarm_code": "ALM-TEMP-402", "severity": "WARNING", "repeat_count": 3},
    {"timestamp": "2026-05-20 10:15", "equipment_id": "TF-CVD-01", "alarm_code": "ALM-TEMP-402", "severity": "CRITICAL", "repeat_count": 5},
    {"timestamp": "2026-05-20 10:50", "equipment_id": "TF-CVD-01", "alarm_code": "ALM-VAC-210", "severity": "WARNING", "repeat_count": 1},
    {"timestamp": "2026-05-20 11:05", "equipment_id": "TF-PVD-02", "alarm_code": "ALM-THK-305", "severity": "WARNING", "repeat_count": 1},
]
# len()은 리스트에 담긴 항목(여기서는 알람 건수) 개수를 세는 함수입니다.
print("전체 로그 건수:", len(alarm_logs))

In [ ]:
# 알람 매뉴얼입니다. 여러 줄에 걸친 긴 문자열은 큰따옴표 3개(""" ... """)로 감싸 작성합니다.
# 아래 내용은 모두 교육용 가상 매뉴얼이며, 실제 설비명·공정 조건·알람 코드와는 무관합니다. (원본에서는 docs/alarm_manual.md 파일을 읽습니다.)
alarm_manual = """[ALM-TEMP-402] 챔버 온도 편차 알람 (박막 증착 공정)
- 의미: 증착 챔버 온도가 기준 범위에서 벗어났을 때 발생합니다.
- 확인: 온도 센서 상태, 챔버 가열/냉각 동작, 최근 온도 추이를 확인합니다.
- 조치: 반복 발생 시 담당 엔지니어에게 상황을 전달합니다.

[ALM-VAC-210] 진공 압력 불안정 알람
- 확인: 진공 압력이 일정한지, 온도 알람과 함께 발생했는지 확인합니다.

[ALM-THK-305] 박막 두께 편차 알람
- 확인: 박막 두께 측정값 추이와 증착 조건 변화 여부를 확인합니다."""
print(alarm_manual[:40], "...")    # 문자열[:40] = 앞 40글자만 잘라 미리 보기

## LLM 해설: 가상 박막 제조 알람 데이터 준비

> 이 셀은 선택 실행 셀입니다. LLM API 설정이 되어 있지 않거나 강의 시간이 부족하면 건너뛰어도 됩니다.

이번 LLM 해설에서는 다음 관점으로 설명을 받습니다.
- 박막 제조 알람 데이터가 필요한 이유
- 입력 데이터가 Agent 판단에 어떻게 쓰이는지
- 주요 변수(sample_query, alarm_logs, alarm_manual)의 역할
- 다음 Agent 판단 단계와의 연결

이 prompt는 `templates/day1`의 `*prompt.mustache` 구조(역할·가상 데이터 안내·입력 정보·작성 요청·제약 조건)를 참고해 만들었으며, 노트북에서는 mustache 원문을 그대로 보여주지 않고 가상의 디스플레이 박막 제조 상황으로 단순화했습니다.

In [ ]:
# 데이터가 길 수 있으므로 preview_text로 줄여서 prompt에 넣습니다.
alarm_data_context = preview_text({
    "sample_query": sample_query,
    "alarm_logs_앞3건": alarm_logs[:3],
    "alarm_manual_일부": alarm_manual[:300],
})

prompt = f"""# 역할
당신은 가상의 디스플레이 박막 제조사 'DisplayEdu ThinFilm Co.'의 AI Agent 교육 강사입니다.

# 교육용 가상 데이터 안내
이 실습은 실제 회사 데이터가 아니라 교육용 가상 박막 제조 시나리오입니다.
실제 사내 데이터·설비명·라인명·알람 코드·공정 조건과 무관합니다.

# 실습 블록
Agent가 판단할 수 있도록 박막 제조 알람 상황 데이터(요청/로그/매뉴얼)를 준비하는 과정입니다.

# 입력 정보
{alarm_data_context}

# 작성 요청
초보자도 이해하도록 5~8줄로 짧게 설명해 주세요.
1. 박막 공정에서 챔버 온도·진공 압력·박막 두께 값이 중요하다는 점
2. sample_query가 공정 엔지니어의 질문 역할을 한다는 점
3. alarm_logs가 가상 설비의 알람 이력이라는 점
4. alarm_manual이 Agent가 참고할 교육용 조치 매뉴얼이라는 점
5. 이 데이터들이 다음 Agent 판단 단계의 재료가 된다는 점

# 제약 조건
- 실제 현장 조치 지시처럼 단정하지 마세요.
- 입력 데이터와 매뉴얼에 없는 내용을 추측하지 마세요.
- "가능성이 있습니다", "확인할 수 있습니다", "추가 확인이 필요합니다" 같은 교육용 표현을 사용하세요.
- 실제 회사명·설비 제조사명·공정 레시피·품질 기준은 넣지 마세요.
- 긴 보고서가 아니라 짧은 강의 해설로 작성하세요."""

# TODO 1: ask_llm 함수를 호출해 위에서 만든 prompt에 대한 LLM 응답을 받으세요.
# TODO 2: title 인자에 이 설명이 무엇인지 알 수 있는 제목을 넣으세요.
# TODO 3: display(Markdown(...)) 으로 응답을 화면에 보기 좋게 출력하세요.
# 힌트: llm_explanation_data = ask_llm(prompt, title="...")
#       display(Markdown(llm_explanation_data))
# 여기에 LLM 호출 코드를 작성하세요.
llm_explanation_data = None


## 3단계. 함수로 알람 로그 조회하고 요약하기

**원본 파일**
- `src/day1/first_sample_run.py` (로그 필터링·요약 부분)

**원본 파일의 목적**
- 설비 ID와 알람 코드가 일치하는 로그만 골라 건수·시각·심각도를 요약합니다.

**이 단원에서 학습할 핵심**
- "데이터를 받아 정해진 일을 하고 결과를 돌려주는" 함수의 기본 구조를 이해합니다.

**노트북에서 사용한 핵심 코드**
- 원본의 필터링/요약 로직을 `search_logs`, `summarize_logs` 두 함수로 옮겼습니다.

**노트북에서 단순화한 부분**
- 원본은 `pandas`로 표를 다루지만, 노트북에서는 순수 Python 반복문(for)으로 같은 일을 합니다.

**단순화한 이유**
- pandas를 모르는 초보자도 한 줄씩 따라 읽을 수 있게 하기 위해서입니다.

**원본 파일을 나중에 볼 때 추가로 확인할 부분**
- `logs[(logs["equipment_id"] == ...) & (logs["alarm_code"] == ...)]` 같은 pandas 필터 문법을 확인하세요.

In [ ]:
# def: 자주 쓰는 작업을 이름 붙여 묶어 둔 "함수" 정의입니다. 한 번 만들어 두면 여러 번 호출할 수 있습니다.
# search_logs 함수는 설비 ID와 알람 코드가 "모두" 일치하는 로그만 골라 돌려줍니다.
# (= Agent가 판단할 근거를 찾기 위해 과거 알람 기록을 조회하는 과정입니다. 원본의 pandas 필터링을 순수 Python으로 바꾼 형태)
def search_logs(logs, equipment_id, alarm_code):
    # TODO 1: 조건에 맞는 로그를 담아 둘 빈 리스트 found 를 만드세요.  (힌트: found = [])
    # TODO 2: for 반복문으로 logs 안의 로그를 한 줄(row)씩 꺼내 확인하세요.  (힌트: for row in logs:)
    # TODO 3: if 문으로 row["equipment_id"] 와 row["alarm_code"] 가 둘 다(and) 일치하는지 비교하세요.
    # TODO 4: 조건에 맞으면 found.append(row) 로 결과 목록에 추가하세요.
    # 여기에 로그 조회 코드를 작성하세요.
    found = []
    return found                           # return: 함수가 처리한 결과(found)를 호출한 쪽으로 돌려줍니다.


In [ ]:
# 위 함수를 실제로 실행해 봅니다.
found_logs = search_logs(alarm_logs, sample_query["equipment_id"], sample_query["alarm_code"])
print("조건에 맞는 로그 건수:", len(found_logs))
for row in found_logs:
    print(" -", row["timestamp"], row["severity"])

In [ ]:
# summarize_logs: 골라낸 로그를 사람이 읽기 쉽게 줄여 요약하는 함수입니다. (건수, 처음/마지막 시각, 심각도별 개수)
# = 여러 알람 로그 중 중요한 내용만 추려 Agent가 한눈에 보기 쉽게 만드는 과정입니다.
def summarize_logs(found_logs):
    # TODO 1: found_logs 가 비어 있으면(if not found_logs) 빈 요약을 반환하세요.
    #         힌트: return {"total_count": 0, "severity_counts": {}}
    # TODO 2: 심각도별 개수를 셀 빈 딕셔너리 severity_counts 를 만드세요.
    # TODO 3: for 문으로 로그를 한 건씩 꺼내며 같은 심각도가 나올 때마다 1씩 더하세요.
    #         힌트: sev = row["severity"] ; severity_counts[sev] = severity_counts.get(sev, 0) + 1
    # TODO 4: total_count, first_time, last_time, severity_counts 를 딕셔너리로 묶어 반환하세요.
    #         힌트: 건수는 len(found_logs), 처음 시각은 found_logs[0]["timestamp"], 마지막은 found_logs[-1]["timestamp"]
    # 여기에 요약 코드를 작성하세요.
    summary = {"total_count": 0, "severity_counts": {}}
    return summary


In [ ]:
# 요약 함수를 실행해 결과를 확인합니다.
log_summary = summarize_logs(found_logs)
print(log_summary)

## LLM 해설: 알람 로그 조회와 요약

> 이 셀은 선택 실행 셀입니다. LLM API 설정이 되어 있지 않거나 강의 시간이 부족하면 건너뛰어도 됩니다.

이번 LLM 해설에서는 다음 관점으로 설명을 받습니다.
- search_logs / summarize_logs 함수의 역할
- 조건에 맞는 알람을 찾는 이유
- 박막 라인에서 반복 알람을 확인하는 의미
- 이 결과가 Agent 판단에 쓰이는 방식

이 prompt는 `templates/day1`의 `*prompt.mustache` 구조(역할·가상 데이터 안내·입력 정보·작성 요청·제약 조건)를 참고해 만들었으며, 노트북에서는 mustache 원문을 그대로 보여주지 않고 가상의 디스플레이 박막 제조 상황으로 단순화했습니다.

In [ ]:
# 로그 조회 결과(found_logs)와 요약(log_summary)을 LLM 해설용 입력으로 정리합니다.
log_search_context = preview_text({"found_logs": found_logs, "log_summary": log_summary})

prompt = f"""# 역할
당신은 가상의 디스플레이 박막 제조사 'DisplayEdu ThinFilm Co.'의 AI Agent 교육 강사입니다.

# 교육용 가상 데이터 안내
이 실습은 실제 회사 데이터가 아니라 교육용 가상 박막 제조 시나리오입니다.
실제 사내 데이터·설비명·라인명·알람 코드·공정 조건과 무관합니다.

# 실습 블록
여러 알람 로그 중 조건에 맞는 것을 찾아(search_logs) 요약(summarize_logs)하는 과정입니다.

# 실행 결과
{log_search_context}

# 작성 요청
초보자도 이해하도록 5~8줄로 짧게 설명해 주세요.
1. 박막 라인에서 같은 알람이 반복되면 공정 이상 징후일 수 있다는 점
2. search_logs 함수가 필요한 알람 기록만 찾는 역할이라는 점
3. summarize_logs 함수가 찾은 로그를 Agent가 보기 쉽게 요약한다는 점
4. 온도 편차·진공 압력·박막 두께 같은 알람을 확인하는 이유
5. 이 요약 결과가 다음 Agent 판단에 어떻게 연결되는지

# 제약 조건
- 실제 현장 조치 지시처럼 단정하지 마세요.
- 로그와 매뉴얼에 없는 내용을 추측하지 마세요.
- "가능성이 있습니다", "확인할 수 있습니다", "추가 확인이 필요합니다" 같은 교육용 표현을 사용하세요.
- 실제 회사명·설비 제조사명·공정 레시피·품질 기준은 넣지 마세요.
- 긴 보고서가 아니라 짧은 강의 해설로 작성하세요."""

# TODO 1: ask_llm 함수를 호출해 위에서 만든 prompt에 대한 LLM 응답을 받으세요.
# TODO 2: title 인자에 이 설명이 무엇인지 알 수 있는 제목을 넣으세요.
# TODO 3: display(Markdown(...)) 으로 응답을 화면에 보기 좋게 출력하세요.
# 힌트: llm_explanation_log_search = ask_llm(prompt, title="...")
#       display(Markdown(llm_explanation_log_search))
# 여기에 LLM 호출 코드를 작성하세요.
llm_explanation_log_search = None


## 4단계. 교육용 Mock LLM 준비하기 (API Key 없이 실행)

**원본 파일**
- `src/llm_client.py` (LLM 호출 공통 창구), `src/llm/mock_llm.py` (교육용 Mock 응답)

**원본 파일의 목적**
- 코드가 특정 LLM 제품에 직접 의존하지 않도록, `generate_response(prompt)` 하나로만 LLM을 부릅니다.
- `.env` 설정에 따라 mock / local(Ollama) / cloud(Gemini·OpenAI 등) 중 하나가 선택됩니다.

**이 단원에서 학습할 핵심**
- Agent는 "Prompt(지시문)를 보내면 답변 글을 돌려받는" 창구 하나만 알면 됩니다.

**노트북에서 사용한 핵심 코드**
- 원본과 같은 이름의 `generate_response(prompt)` 함수를 아주 작은 Mock 형태로 다시 만듭니다.

**노트북에서 단순화한 부분**
- provider 분기(mock/local/cloud), `.env` 읽기, 실제 API 호출, 예외 처리, 보안 마스킹을 모두 제외했습니다.
- 프롬프트에 들어 있는 알람 코드를 보고 정해진 답변을 돌려주는 방식만 남겼습니다.

**단순화한 이유**
- 외부 API Key나 인터넷 없이도 누구나 즉시 실행할 수 있게 하기 위해서입니다.

**원본 파일을 나중에 볼 때 추가로 확인할 부분**
- `llm_client.generate_response`가 어떻게 mock/local/cloud를 고르는지, 호출 실패 시 mock으로 어떻게 전환되는지 확인하세요.

### 초보자 관점에서 이해하기
이 노트북에는 LLM을 부르는 함수가 **두 가지**라 헷갈릴 수 있어 정리합니다.
- `generate_response(...)` — 이 셀에서 만드는 **교육용 가짜(Mock) LLM**입니다. 핵심 실습(Chain·Agent v0)에서 API Key 없이도 항상 같은 답을 돌려주도록 사용합니다.
- `ask_llm(...)` — 맨 앞 공통 셀에서 만든 **진짜 LLM 호출 함수**입니다. 선택 실행되는 "LLM 해설" 셀에서만 사용합니다.

In [ ]:
# 교육용 Mock(가짜) LLM입니다. 실제 LLM 서버를 부르지 않고, 미리 정해 둔 답변을 돌려줍니다.
# (LLM = 질문에 답하는 언어 모델, prompt = LLM에게 주는 업무 지시서입니다. 여기서는 API Key 없이도 실습되도록 가짜로 흉내만 냅니다.)
def generate_response(prompt):
    if "ALM-TEMP-402" in prompt:           # prompt 안에 이 알람 코드가 들어 있으면(in) 그에 맞는 답을 고릅니다.
        return ("## 온도 편차 알람 분석 (교육용 가상 응답)\n"
                "- 원인 후보: 온도 제어 불안정, 센서 측정값 변동 가능성\n"
                "- 1차 확인: 온도 센서, 냉각 장치, 최근 온도 추이\n"
                "- 권장: 반복 발생 시 담당 엔지니어에게 전달")
    return ("## 분석 결과 (교육용 가상 응답)\n"   # 그 외에는 기본 안내 답변을 돌려줍니다.
            "- 입력 정보를 더 확인해야 정확한 분석이 가능합니다.")

In [ ]:
# 프롬프트에 알람 코드가 들어 있으면, 그에 맞는 답변을 돌려줍니다.
print(generate_response("ALM-TEMP-402 알람을 분석해 주세요."))

## 5단계. 간단한 Chain 만들기 (매뉴얼 검색 → Prompt → LLM)

**원본 파일**
- `src/day1/simple_chain_starter.py`

**원본 파일의 목적**
- "사용자 요청 → 매뉴얼 검색 → Prompt 생성 → LLM 호출 → 결과 정리"처럼 여러 단계를 순서대로 잇는 Chain 흐름을 보여 줍니다.

**이 단원에서 학습할 핵심**
- Chain은 컨베이어 벨트처럼, 한 단계의 결과가 다음 단계의 입력이 되는 구조입니다.

**노트북에서 사용한 핵심 코드**
- 매뉴얼 단순 검색 `search_manual`, 프롬프트 생성 `build_prompt`를 만들고, 4단계의 `generate_response`로 이어 붙입니다.

**노트북에서 단순화한 부분**
- 원본은 Mustache 템플릿 파일(`*.mustache`)을 `pystache`로 렌더링하지만, 노트북에서는 f-string으로 Prompt를 만듭니다.

**단순화한 이유**
- 템플릿 파일을 따로 열지 않고도 Prompt가 어떻게 만들어지는지 한 셀에서 바로 보이게 하기 위해서입니다.

**원본 파일을 나중에 볼 때 추가로 확인할 부분**
- `pystache.render(template, data)`로 `templates/day1/*.mustache`를 채우는 부분을 확인하세요.

In [ ]:
# 매뉴얼에서 알람 코드나 핵심 단어가 들어 있는 "줄"만 골라내는 단순 검색 함수입니다.
# 단어가 들어 있는지만 보는 검색이라, 의미가 비슷한 다른 표현은 찾지 못합니다.
# (외부 문서를 의미 기반으로 찾아본 뒤 그 내용을 근거로 답하는 방식인 RAG는 2일차에 배웁니다.)
def search_manual(manual_text, alarm_code):
    # TODO 1: 찾은 줄을 담을 빈 리스트 found_lines 를 만드세요.
    # TODO 2: manual_text.splitlines() 로 매뉴얼을 줄 단위로 잘라 for 문으로 한 줄(line)씩 확인하세요.
    # TODO 3: if 문으로 alarm_code 나 핵심 단어("온도", "확인", "조치") 중 하나라도 줄에 들어 있는지(in / or) 확인하세요.
    # TODO 4: 조건에 맞는 줄을 found_lines 에 추가(append)하세요.
    # 여기에 매뉴얼 검색 코드를 작성하세요.
    found_lines = []
    return "\n".join(found_lines)          # 골라낸 줄들을 줄바꿈으로 이어 하나의 문자열로 돌려줍니다.


In [ ]:
# build_prompt: 사용자 요청 + 로그 요약 + 매뉴얼 내용을 하나로 합쳐, LLM에게 보낼 Prompt(업무 지시서)를 만듭니다.
# f"""...""" 는 f-string으로, 문장 안의 {중괄호}에 변수 값을 그대로 끼워 넣어 주는 Python 문법입니다.
# (원본은 Mustache 템플릿 파일을 쓰지만, 여기서는 흐름이 한눈에 보이도록 f-string으로 직접 작성했습니다.)
def build_prompt(query, log_summary, manual_section):
    return f"""당신은 제조 알람 분석 보조 Agent입니다.
[사용자 요청] {query['user_query']}
[설비] {query['equipment_id']} / [알람] {query['alarm_code']}
[로그 요약] {log_summary}
[매뉴얼 참고]
{manual_section}

위 정보를 바탕으로 원인 후보와 1차 확인 항목을 정리해 주세요."""

In [ ]:
# Chain 실행: 매뉴얼 검색 -> Prompt 생성 -> LLM 호출을 순서대로 잇습니다.
# 컨베이어 벨트처럼, 앞 단계의 결과(manual_section)가 다음 단계(build_prompt)의 입력이 됩니다.
# TODO 1: search_manual(alarm_manual, sample_query["alarm_code"]) 로 관련 매뉴얼 줄을 찾아 manual_section 에 담으세요.
# TODO 2: build_prompt(sample_query, log_summary, manual_section) 로 LLM에 보낼 prompt 를 만드세요.
# TODO 3: generate_response(prompt) 로 LLM(여기서는 교육용 Mock) 응답을 response 에 담으세요.
# 여기에 Chain 코드를 작성하세요.
manual_section = ""
prompt = ""
response = ""

print("[생성된 Prompt 일부]")
print(prompt[:120], "...\n")     # prompt[:120] = 만들어진 지시서의 앞 120글자만 잘라 미리 보기
print("[LLM 응답]")
print(response)


## 6단계. 가장 단순한 Agent v0 만들기

**원본 파일**
- `src/day1/day1_agent_v0_template.py` (전체 통합), `src/day1/mini_graph_runner.py` (조건부 분기)

**원본 파일의 목적**
- 앞에서 따로 배운 데이터 조회·매뉴얼 검색·Prompt·LLM 호출을 하나로 합쳐, "스스로 판단하는" 가장 단순한 Agent를 만듭니다.

**이 단원에서 학습할 핵심**
- Agent는 거창한 AI가 아니라, **입력을 보고 다음 행동을 정하는 판단 흐름**입니다.
- 여기서는 "필수 정보가 있으면 조사하고, 없으면 정보를 더 요청"하는 갈림길(조건부 분기)을 만듭니다.

**노트북에서 사용한 핵심 코드**
- 원본의 여러 Node(처리 단계)를 `run_agent_v0` 함수 하나로 모아, if 문으로 분기합니다.

**노트북에서 단순화한 부분**
- 원본은 `langgraph`의 StateGraph/Node/Conditional Edge로 흐름을 구성하지만, 노트북에서는 일반 함수와 if 문으로 같은 판단을 표현했습니다.
- 결과를 파일로 저장하는 부분은 제외했습니다.

**단순화한 이유**
- LangGraph를 처음 배우기 전에, Agent의 "판단" 자체를 먼저 이해하도록 하기 위해서입니다.

**원본 파일을 나중에 볼 때 추가로 확인할 부분**
- `StateGraph`, `add_node`, `add_conditional_edges`로 흐름을 그리는 부분을 확인하세요. (다음 9단계에서 개념을 미리 맛봅니다.)

In [ ]:
# Agent(= 상황을 보고 다음 행동을 스스로 고르는 작업자)의 가장 단순한 형태입니다.
# 입력을 보고 "조사할지 / 정보를 더 요청할지"를 if 문으로 판단합니다. (return이 2개 = 두 갈래 길)
def run_agent_v0(query, logs, manual_text):
    # (1) 판단: 필수 정보(설비 ID, 알람 코드)가 없으면, 무리하게 분석하지 않고 정보를 더 요청하고 끝냅니다.
    # TODO 1: query.get("equipment_id") 와 query.get("alarm_code") 중 하나라도 비어 있으면
    #         {"action": "ask_more_info", "message": "분석하려면 equipment_id와 alarm_code가 필요합니다."} 를 반환하세요.
    # (2) 조사: 필수 정보가 있으면 로그 조회 -> 요약 -> 매뉴얼 검색을 차례로 수행합니다.
    # TODO 2: search_logs(logs, query["equipment_id"], query["alarm_code"]) 로 found 를,
    #         summarize_logs(found) 로 summary 를, search_manual(manual_text, query["alarm_code"]) 로 manual_section 을 구하세요.
    # (3) 정리: 모은 정보로 Prompt를 만들어 LLM에 보내고, 그 답을 결과에 담아 돌려줍니다.
    # TODO 3: generate_response(build_prompt(query, summary, manual_section)) 로 answer 를 구하세요.
    # TODO 4: {"action": "investigate", "summary": summary, "response": answer} 를 반환하세요.
    # 여기에 Agent 판단 코드를 작성하세요.
    result = {"action": None, "summary": {}, "response": "", "message": ""}
    return result


## 7단계. Agent 실행 결과 확인하기

이제 6단계에서 만든 `run_agent_v0`를 실제로 실행해 봅니다.

- 필수 정보가 모두 있는 **정상 케이스**에서는 Agent가 로그를 조사하고 답변을 만듭니다.
- 반환값의 `action`이 `"investigate"`이면 "조사를 진행했다"는 뜻입니다.

In [ ]:
# 정상 케이스: sample_query에는 설비 ID와 알람 코드가 모두 들어 있습니다.
result = run_agent_v0(sample_query, alarm_logs, alarm_manual)

print("Agent 행동:", result["action"])
print("로그 요약:", result["summary"])
print("\n[Agent 답변]")
print(result["response"])

## LLM 해설: Agent 판단 결과

> 이 셀은 선택 실행 셀입니다. LLM API 설정이 되어 있지 않거나 강의 시간이 부족하면 건너뛰어도 됩니다.

이번 LLM 해설에서는 다음 관점으로 설명을 받습니다.
- 여기서 Agent가 하는 일
- 입력 데이터와 판단 결과의 관계
- 새 지식을 만든 것이 아니라 주어진 정보를 정리했다는 점
- 이 구조의 한계와 2일차 RAG 연결

이 prompt는 `templates/day1`의 `*prompt.mustache` 구조(역할·가상 데이터 안내·입력 정보·작성 요청·제약 조건)를 참고해 만들었으며, 노트북에서는 mustache 원문을 그대로 보여주지 않고 가상의 디스플레이 박막 제조 상황으로 단순화했습니다.

In [ ]:
# Agent 실행 결과(result)를 LLM 해설용 입력으로 정리합니다.
agent_result_context = preview_text(result)

prompt = f"""# 역할
당신은 가상의 디스플레이 박막 제조사 'DisplayEdu ThinFilm Co.'의 AI Agent 교육 강사입니다.

# 교육용 가상 데이터 안내
이 실습은 실제 회사 데이터가 아니라 교육용 가상 박막 제조 시나리오입니다.
실제 사내 데이터·설비명·라인명·알람 코드·공정 조건과 무관합니다.

# 실습 블록
박막 제조 알람 상황을 입력받아 Agent가 원인 후보와 확인 항목을 정리하는 과정입니다.

# 실행 결과
{agent_result_context}

# 작성 요청
초보자도 이해하도록 5~8줄로 짧게 설명해 주세요.
1. Agent가 박막 공정 엔지니어의 보조 역할을 한다는 점
2. 입력 데이터·알람 로그·매뉴얼이 Agent 판단에 어떻게 쓰이는지
3. Agent가 새로운 사실을 만든 것이 아니라 주어진 정보를 정리한다는 점
4. Agent v0는 실제 원인을 단정하지 않는다는 점
5. 2일차 RAG에서 매뉴얼 검색이 강화된다는 점

# 제약 조건
- 실제 현장 조치 지시처럼 단정하지 마세요.
- 입력 데이터와 매뉴얼에 없는 내용을 추측하지 마세요.
- "가능성이 있습니다", "확인할 수 있습니다", "추가 확인이 필요합니다" 같은 교육용 표현을 사용하세요.
- 실제 회사명·설비 제조사명·공정 레시피·품질 기준은 넣지 마세요.
- 긴 보고서가 아니라 짧은 강의 해설로 작성하세요."""

# TODO 1: ask_llm 함수를 호출해 위에서 만든 prompt에 대한 LLM 응답을 받으세요.
# TODO 2: title 인자에 이 설명이 무엇인지 알 수 있는 제목을 넣으세요.
# TODO 3: display(Markdown(...)) 으로 응답을 화면에 보기 좋게 출력하세요.
# 힌트: llm_explanation_agent_result = ask_llm(prompt, title="...")
#       display(Markdown(llm_explanation_agent_result))
# 여기에 LLM 호출 코드를 작성하세요.
llm_explanation_agent_result = None


## 8단계. 여러 제조 상황을 테스트해 보기

같은 Agent라도 입력이 달라지면 행동이 달라집니다. 두 가지 상황을 확인합니다.

1. **정보 부족 케이스**: 설비 ID가 비어 있으면 Agent는 조사 대신 "정보를 더 달라"고 합니다.
2. **로그가 없는 케이스**: 매칭되는 로그가 없으면 요약 건수가 0이 됩니다.

이 부분은 원본 `mini_graph_runner.py`의 "정상 케이스 / 정보 부족 케이스" 비교와 같은 아이디어입니다.

In [ ]:
# 정보 부족 케이스: equipment_id를 빈 값("")으로 두어 필수 정보를 일부러 비웁니다.
# 데이터가 부족할 때 Agent가 무리하게 단정하지 않고 "정보를 더 달라"고 되묻는지 확인하는 실습입니다.
missing_query = {
    "user_query": "박막 증착 공정에서 온도 알람이 난 것 같은데 어떤 정보가 더 필요한가요?",
    "equipment_id": "",            # 비어 있음
    "alarm_code": "ALM-TEMP-402",
}
missing_result = run_agent_v0(missing_query, alarm_logs, alarm_manual)
print("Agent 행동:", missing_result["action"])     # ask_more_info 가 나오면 되묻기 쪽으로 갈라진 것입니다.
print("안내 메시지:", missing_result["message"])

In [ ]:
# 무매칭 케이스: 존재하지 않는 설비 ID로 조회해, 조건에 맞는 로그가 하나도 없을 때를 확인하는 실습입니다.
# dict(sample_query, equipment_id="TF-XX-99") = 기존 요청을 복사하되 설비 ID만 바꿔 새 dict를 만드는 방법입니다.
nomatch_query = dict(sample_query, equipment_id="TF-XX-99")
nomatch_result = run_agent_v0(nomatch_query, alarm_logs, alarm_manual)
print("Agent 행동:", nomatch_result["action"])     # 필수 정보는 있으므로 investigate 로 가지만,
print("로그 요약:", nomatch_result["summary"])      # 맞는 로그가 없어 total_count는 0이 됩니다.

## LLM 해설: 여러 박막 제조 상황 테스트

> 이 셀은 선택 실행 셀입니다. LLM API 설정이 되어 있지 않거나 강의 시간이 부족하면 건너뛰어도 됩니다.

이번 LLM 해설에서는 다음 관점으로 설명을 받습니다.
- 같은 Agent라도 입력에 따라 행동이 달라진다는 점
- 정보 부족 시 되묻는 동작의 의미
- 매칭 로그가 없을 때의 처리
- 이런 분기가 박막 제조 현장에서 중요한 이유

이 prompt는 `templates/day1`의 `*prompt.mustache` 구조(역할·가상 데이터 안내·입력 정보·작성 요청·제약 조건)를 참고해 만들었으며, 노트북에서는 mustache 원문을 그대로 보여주지 않고 가상의 디스플레이 박막 제조 상황으로 단순화했습니다.

In [ ]:
# 정보 부족 / 무매칭 케이스 결과를 함께 정리합니다.
test_cases_context = preview_text({"정보부족_케이스": missing_result, "무매칭_케이스": nomatch_result})

prompt = f"""# 역할
당신은 가상의 디스플레이 박막 제조사 'DisplayEdu ThinFilm Co.'의 AI Agent 교육 강사입니다.

# 교육용 가상 데이터 안내
이 실습은 실제 회사 데이터가 아니라 교육용 가상 박막 제조 시나리오입니다.
실제 사내 데이터·설비명·라인명·알람 코드·공정 조건과 무관합니다.

# 실습 블록
같은 Agent에 서로 다른 박막 제조 상황(정보 부족 / 매칭 로그 없음)을 넣어 본 결과입니다.

# 실행 결과
{test_cases_context}

# 작성 요청
초보자도 이해하도록 5~8줄로 짧게 설명해 주세요.
1. 같은 Agent라도 입력이 다르면 행동이 달라진다는 점
2. 필수 정보가 없을 때 되묻는 동작의 의미
3. 매칭되는 로그가 없을 때의 처리
4. 이런 분기 처리가 박막 제조 현장에서 중요한 이유

# 제약 조건
- 실제 현장 조치 지시처럼 단정하지 마세요.
- 입력 데이터에 없는 내용을 추측하지 마세요.
- "가능성이 있습니다", "확인할 수 있습니다", "추가 확인이 필요합니다" 같은 교육용 표현을 사용하세요.
- 실제 회사명·설비 제조사명·공정 레시피·품질 기준은 넣지 마세요.
- 긴 보고서가 아니라 짧은 강의 해설로 작성하세요."""

# TODO 1: ask_llm 함수를 호출해 위에서 만든 prompt에 대한 LLM 응답을 받으세요.
# TODO 2: title 인자에 이 설명이 무엇인지 알 수 있는 제목을 넣으세요.
# TODO 3: display(Markdown(...)) 으로 응답을 화면에 보기 좋게 출력하세요.
# 힌트: llm_explanation_test_cases = ask_llm(prompt, title="...")
#       display(Markdown(llm_explanation_test_cases))
# 여기에 LLM 호출 코드를 작성하세요.
llm_explanation_test_cases = None


## 9단계. State / Node / Graph 기초 살짝 맛보기

**원본 파일**
- `src/day1/mini_graph_runner.py`

**원본 파일의 목적**
- LangGraph로 State(작업 기록지), Node(처리 단계), Conditional Edge(조건부 갈림길)를 구성하는 법을 보여 줍니다.

**이 단원에서 학습할 핵심**
- **State**: 작업하며 계속 채워 나가는 기록지 (비유: 상담원이 적어 두는 상담 기록지)
- **Node**: 흐름 중 하나의 처리 단계 (비유: 공정 라인의 한 작업 구간)
- **Graph**: 여러 Node를 연결한 전체 순서도 (비유: 공장의 작업 순서도)

**노트북에서 사용한 핵심 코드**
- `state`라는 dict를 만들고, 그 state를 받아 채우는 작은 Node 함수들을 순서대로 실행합니다.

**노트북에서 단순화한 부분**
- 실제 `langgraph` 라이브러리(StateGraph, add_node, add_conditional_edges)는 쓰지 않았습니다.
- 대신 순수 Python 함수 호출 순서로 "Node를 지나며 State가 채워지는 느낌"만 보여 줍니다.

**단순화한 이유**
- 라이브러리 사용법보다 State/Node 개념을 먼저 직관적으로 이해하도록 하기 위해서입니다.

**원본 파일을 나중에 볼 때 추가로 확인할 부분**
- `graph.add_node(...)`, `graph.add_conditional_edges(...)`, `app.invoke(state)` 부분을 확인하세요.

In [ ]:
# State = Agent가 지금 알고 있는 정보를 담는 "작업 메모장"(dict)입니다.
# Node = 그 메모장(state)을 받아 한 단계 작업을 하고 내용을 채워 돌려주는 함수(작업 단계)입니다.
def node_check_info(state):                # Node 1: 필수 정보가 있는지 확인하는 작업 단계
    q = state["query"]
    # TODO 1: equipment_id 와 alarm_code 가 모두 있으면 state["next_action"] 을 "search_log" 로,
    #         아니면 "ask_more_info" 로 정하세요.  (힌트: state["next_action"] = "search_log" if 조건 else "ask_more_info")
    # TODO 2: state["trace"] 에 "check_info" 를 append 해 거쳐 간 단계를 기록하세요.
    # 여기에 코드를 작성하세요.
    return state                           # 내용을 채운 메모장을 돌려줍니다.

def node_search_log(state):                # Node 2: 로그를 조회해 결과를 메모장에 담는 작업 단계
    q = state["query"]
    # TODO 1: search_logs(alarm_logs, q["equipment_id"], q["alarm_code"]) 결과를 state["found"] 에 담으세요.
    # TODO 2: state["trace"] 에 "search_log" 를 append 하세요.
    # 여기에 코드를 작성하세요.
    return state


In [ ]:
# Graph = 여러 Node가 어떤 순서로 실행될지 정리한 "작업 순서도"입니다. 여기서는 함수 호출 순서로 흉내 냅니다.
# 먼저 빈 작업 메모장(State)을 만듭니다. (trace=거쳐 간 단계 기록, found=찾은 로그, next_action=다음 행동)
state = {"query": sample_query, "trace": [], "found": [], "next_action": ""}

# TODO 1: node_check_info(state) 를 호출해 1번째 Node를 지나며 state 를 채우세요. (결과를 다시 state 에 담기)
# TODO 2: state["next_action"] 이 "search_log" 일 때만(if) node_search_log(state) 로 2번째 Node를 실행하세요.
#         (조건부 분기 = 다음 행동에 따라 갈 길이 달라지는 Conditional Edge 흉내)
# 여기에 Graph 실행 코드를 작성하세요.

print("거쳐 간 Node:", state["trace"])
print("다음 행동:", state["next_action"])
print("찾은 로그 건수:", len(state["found"]))


## LLM 해설: State / Node / Graph 기초

> 이 셀은 선택 실행 셀입니다. LLM API 설정이 되어 있지 않거나 강의 시간이 부족하면 건너뛰어도 됩니다.

이번 LLM 해설에서는 다음 관점으로 설명을 받습니다.
- State(작업 기록지)의 의미
- Node(처리 단계)가 State를 채운다는 점
- 조건부 분기로 경로가 달라진다는 점
- 이것이 LangGraph 개념과 어떻게 이어지는지

이 prompt는 `templates/day1`의 `*prompt.mustache` 구조(역할·가상 데이터 안내·입력 정보·작성 요청·제약 조건)를 참고해 만들었으며, 노트북에서는 mustache 원문을 그대로 보여주지 않고 가상의 디스플레이 박막 제조 상황으로 단순화했습니다.

In [ ]:
# State에 기록된 trace / 다음 행동 / 찾은 로그 수를 정리합니다.
graph_context = preview_text({
    "거쳐간_Node": state["trace"],
    "다음_행동": state["next_action"],
    "찾은_로그_수": len(state["found"]),
})

prompt = f"""# 역할
당신은 가상의 디스플레이 박막 제조사 'DisplayEdu ThinFilm Co.'의 AI Agent 교육 강사입니다.

# 교육용 가상 데이터 안내
이 실습은 실제 회사 데이터가 아니라 교육용 가상 박막 제조 시나리오입니다.
실제 사내 데이터·설비명·라인명·알람 코드·공정 조건과 무관합니다.

# 실습 블록
State(작업 기록지)를 Node(처리 단계)들이 차례로 채우고, 조건에 따라 다음 단계가 갈라지는 흐름입니다.

# 실행 결과
{graph_context}

# 작성 요청
초보자도 이해하도록 5~8줄로, 어려운 용어는 박막 공장 비유로 풀어 설명해 주세요.
1. State는 박막 설비 점검 기록지처럼 이해할 수 있다는 점
2. Node는 입력 확인·로그 조회·판단 같은 하나의 처리 단계라는 점
3. Graph는 이런 단계가 연결된 작업 순서도라는 점
4. 조건에 따라 다음 단계가 달라지는 이유
5. 이 개념이 나중에 LangGraph 학습으로 이어진다는 점

# 제약 조건
- 실제 현장 조치 지시처럼 단정하지 마세요.
- 실행 결과에 없는 내용을 추측하지 마세요.
- "확인할 수 있습니다", "추가 확인이 필요합니다" 같은 교육용 표현을 사용하세요.
- 실제 회사명·설비 제조사명·공정 레시피·품질 기준은 넣지 마세요.
- 긴 보고서가 아니라 짧은 강의 해설로 작성하세요."""

# TODO 1: ask_llm 함수를 호출해 위에서 만든 prompt에 대한 LLM 응답을 받으세요.
# TODO 2: title 인자에 이 설명이 무엇인지 알 수 있는 제목을 넣으세요.
# TODO 3: display(Markdown(...)) 으로 응답을 화면에 보기 좋게 출력하세요.
# 힌트: llm_explanation_graph = ask_llm(prompt, title="...")
#       display(Markdown(llm_explanation_graph))
# 여기에 LLM 호출 코드를 작성하세요.
llm_explanation_graph = None


## 10단계. Agent v0의 한계 정리

오늘 만든 Agent v0는 잘 동작하지만, 분명한 한계가 있습니다.

1. **미리 정해진 규칙/데이터 안에서만 동작합니다.** 매뉴얼에 없는 새로운 내용을 스스로 찾지 못합니다.
2. **매뉴얼 검색이 "단어 포함" 수준입니다.** 의미가 비슷한 다른 표현은 찾지 못합니다.
3. **LLM이 Mock입니다.** 정해진 답만 돌려주므로 실제로 추론하지는 않습니다.
4. **도구(Tool) 사용이나 외부 시스템 연결이 없습니다.** (Tool 호출/MCP는 3일차 이후에 배웁니다.)

> 참고: 여기서 **Tool**은 "Agent가 직접 실행할 수 있는 기능(조회 버튼, 계산기 등)"을 뜻하고, 여러 도구를 정해진 규칙으로 연결하는 방식이 **MCP**입니다. 둘 다 이후 강의에서 다룹니다.

## 11단계. 2일차 RAG 학습으로 이어지는 이유

Agent v0의 가장 큰 한계는 "매뉴얼을 단어 단위로만 찾는다"는 점이었습니다.

2일차에 배우는 **RAG**는 이 한계를 해결합니다.
- **RAG**: 외부 문서·매뉴얼을 **의미 기반으로 찾아본 뒤** 그 내용을 근거로 답변하는 방식입니다.
- 비유: 기억만으로 답하지 않고, **관련 설명서를 찾아 읽은 뒤** 답하는 방식입니다.

즉, 오늘의 단순 검색(`search_manual`)이 2일차에는 더 똑똑한 검색으로 발전합니다.

## 마무리 정리

이번 노트북에서는 `src/day1`의 핵심 코드를 바탕으로 **가장 단순한 Agent 흐름**을 실습했습니다.
원본 파일 전체를 복사한 것이 아니라, 각 파일의 목적을 이해하는 데 필요한 핵심만 추출했습니다.

**오늘 배운 내용**
1. Python 코드의 위→아래 실행 흐름
2. 제조 알람 데이터(요청/로그/매뉴얼) 다루기
3. 함수로 로그 조회·요약하기
4. Prompt를 만들어 LLM(Mock)에게 보내는 Chain
5. 입력을 보고 판단하는 Agent v0와 조건부 분기
6. State/Node/Graph 개념 맛보기

**사용한 주요 원본 파일과 목적**
- `first_sample_run.py`: 데이터 읽기→LLM 정리 첫 실행
- `simple_chain_starter.py`: 단계를 잇는 Chain
- `mini_graph_runner.py`: State/Node/조건부 분기
- `day1_agent_v0_template.py`: 위 개념을 합친 Agent v0
- `src/llm_client.py` + `src/llm/mock_llm.py`: LLM 호출 공통 창구와 Mock 응답

**노트북에서 단순화한 부분 요약**
- pandas → 순수 Python, Mustache 템플릿 → f-string, LangGraph → 일반 함수+if, 파일 입출력/.env/실제 API 호출 제외

**Agent v0의 한계 → 다음 단계**
- 오늘 Agent는 정해진 규칙·단어 검색 안에서만 동작합니다.
- 2일차 **RAG**로 매뉴얼을 의미 기반으로 찾아 답하도록 확장합니다.

**참고 파일(핵심 실습 제외)**
- `*_streamlit_app.py` 4개: CLI 버전과 로직이 같고 화면 표시만 달라 참고용으로만 소개했습니다.
- `__init__.py`: 실행 코드가 없어 제외했습니다.

> 이 노트북은 `src/day1` 원본 파일을 전혀 수정하지 않았습니다.

## LLM 최종 요약: Day1 학습 정리

> 이 셀은 선택 실행 셀입니다. LLM API 설정이 되어 있지 않거나 강의 시간이 부족하면 건너뛰어도 됩니다.

마지막으로 LLM이 오늘 배운 내용을 다시 요약하고, Agent v0의 한계와 2일차 RAG 연결을 설명하게 합니다.
강의 마무리 또는 복습용으로 사용할 수 있습니다.

이 prompt는 `templates/day1`의 `*prompt.mustache` 구조를 참고했고, 가상의 디스플레이 박막 제조 상황으로 단순화했습니다.

In [ ]:
final_prompt = """# 역할
당신은 가상의 디스플레이 박막 제조사 'DisplayEdu ThinFilm Co.'의 AI Agent 교육 강사입니다.

# 교육용 가상 데이터 안내
이 정리는 실제 회사 데이터가 아니라 교육용 가상 박막 제조 시나리오를 기준으로 합니다.

# 실습 블록
Day1 박막 제조 AI Agent 기초 수업의 전체 내용을 마무리 정리합니다.
다룬 내용: Python 실행 흐름, 박막 알람 데이터 준비, 알람 로그 조회/요약,
Prompt/Chain/LLM 호출, Agent v0 판단 흐름, State/Node/Graph 기초.

# 작성 요청
수강생이 복습하도록 8줄 이내로 짧게 정리해 주세요.
1. 오늘 배운 핵심 3가지
2. Agent v0의 한계
3. 왜 RAG가 필요한지
4. 다음 수업에서 기대할 점

# 제약 조건
- 실제 현장 조치 지시처럼 단정하지 마세요.
- "확인할 수 있습니다", "추가 확인이 필요합니다" 같은 교육용 표현을 사용하세요.
- 실제 회사명·설비 제조사명·공정 레시피·품질 기준은 넣지 마세요.
- 긴 보고서가 아니라 짧은 강의 해설로 작성하세요."""

# TODO 1: ask_llm 함수를 호출해 위에서 만든 prompt에 대한 LLM 응답을 받으세요.
# TODO 2: title 인자에 이 설명이 무엇인지 알 수 있는 제목을 넣으세요.
# TODO 3: display(Markdown(...)) 으로 응답을 화면에 보기 좋게 출력하세요.
# 힌트: llm_final_summary = ask_llm(final_prompt, title="...")
#       display(Markdown(llm_final_summary))
# 여기에 LLM 호출 코드를 작성하세요.
llm_final_summary = None
